# Generic Camouflaged Object Detection (COD) Preprocessing & Tactical Augmentation Pipeline

This notebook provides a **folder-agnostic, flexible, and production-grade TensorFlow (`tf.data.Dataset`) preprocessing & export pipeline** for Camouflaged Object Detection (COD) image datasets (e.g. COD10K, CAMO, NC4K, ACD1K, tactical military surveillance, or custom datasets).

### Key Capabilities & Architecture:
1. **Automated Export to `data/augmented/`**: Includes a dedicated pipeline function `generate_and_save_augmented_dataset()` that generates $N$ augmented RGB variants per image and exports them directly into the `data/augmented/` directory (`images/`, `GT/`, `edges/`).
2. **RGB Image-Only Augmentation**: **ONLY the RGB images undergo tactical data augmentations** (CLAHE contrast adjustment, brightness/contrast jitter, HSV color shifts, weather blur, sensor noise), while Ground Truth masks and Sobel edge maps remain strictly clean, un-augmented ground truth labels.
3. **TensorFlow `tf.data.Dataset` Engine**: Fully integrated with TensorFlow 2.x utilizing `tf.data.Dataset`, `tf.py_function`, and `prefetch(tf.data.AUTOTUNE)` for high-throughput execution.
4. **Dynamic Path Resolution & Auto-Discovery**: Automatically adapts to any root directory structure (`data/raw/dataset-splitM`, `COD10K`, `army_dataset`, or custom paths specified in `params.yaml`).
5. **Extension-Agnostic File Pairing**: Automatically pairs RGB images (`.jpg`, `.png`, `.jpeg`, `.bmp`, `.webp`) and Ground Truth (GT) masks based on filename stems.
6. **Mask Cleaning & Binarization**: Normalizes GT masks into clean binary matrices ($G \in \{0, 1\}$).
7. **Automated Sobel Edge Extraction**: Computes ground-truth Sobel boundary maps ($E \in \{0, 1\}$) required for boundary supervision in dual-branch COD networks.
8. **Interactive Visualizations**: Side-by-side display of original RGB images, augmented RGB images, binary GT masks, Sobel edge maps, and color overlays.

## 1. Imports & Environment Configuration

In [ ]:
import os
import sys
import glob
import re
from pathlib import Path
import yaml
import cv2
import numpy as np
import tensorflow as tf
import albumentations as A
import matplotlib.pyplot as plt
from tqdm import tqdm

print(f"TensorFlow Version: {tf.__version__}")

# Set seed for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Attempt to load configuration from params.yaml if available
PARAMS_FILE = Path("../params.yaml")
if PARAMS_FILE.exists():
    with open(PARAMS_FILE, "r") as f:
        params = yaml.safe_load(f)
    dataset_cfg = params.get("dataset", {})
    IMAGE_SIZE = tuple(dataset_cfg.get("image_size", [384, 384]))
    BATCH_SIZE = dataset_cfg.get("batch_size", 16)
    RAW_DATA_ROOT = Path("../") / dataset_cfg.get("raw_dir", "data/raw")
    AUGMENTED_DATA_ROOT = Path("../data/augmented")
    PROCESSED_DATA_ROOT = Path("../") / dataset_cfg.get("processed_dir", "data/processed")
    print(f"Loaded parameters from {PARAMS_FILE}:")
    print(f"  Image Size: {IMAGE_SIZE}, Batch Size: {BATCH_SIZE}")
    print(f"  Raw Data Root:       {RAW_DATA_ROOT.resolve()}")
    print(f"  Augmented Data Root: {AUGMENTED_DATA_ROOT.resolve()}")
else:
    IMAGE_SIZE = (384, 384)
    BATCH_SIZE = 16
    RAW_DATA_ROOT = Path("../data/raw")
    AUGMENTED_DATA_ROOT = Path("../data/augmented")
    PROCESSED_DATA_ROOT = Path("../data/processed")
    print("Using default configuration settings.")

# PREPROCESSING MODE SELECTION:
# Modes: 'RGB_CLAHE' (Recommended for Tactical COD), 'RGB_STANDARD', 'LAB_COLORSPACE'
COD_PREPROCESSING_MODE = 'RGB_CLAHE'
CUSTOM_DATASET_PATH = None  # Set to a Path object or string if overriding auto-discovery


## 2. Generic Dataset Discovery Engine

In [ ]:
def discover_image_mask_pairs(data_dir, image_subdirs=None, mask_subdirs=None, valid_exts=None):
    """
    Generic auto-discovery engine to locate RGB images and matching GT masks.
    """
    if valid_exts is None:
        valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
    
    if image_subdirs is None:
        image_subdirs = ["images", "image", "imgs", "img", "rgb", "inputs", "src"]
    if mask_subdirs is None:
        mask_subdirs = ["gt", "masks", "mask", "labels", "label", "groundtruth", "ground-truth"]

    data_dir = Path(data_dir)
    if not data_dir.exists():
        raise FileNotFoundError(f"Target path does not exist: {data_dir.resolve()}")

    img_dir, mask_dir = None, None

    for child in data_dir.rglob("*"):
        if child.is_dir():
            folder_name = child.name.lower()
            if img_dir is None and any(folder_name == k for k in image_subdirs):
                img_dir = child
            elif mask_dir is None and any(folder_name == k for k in mask_subdirs):
                mask_dir = child

    if img_dir is None or mask_dir is None:
        for child in data_dir.iterdir():
            if child.is_dir():
                folder_name = child.name.lower()
                if img_dir is None and any(k in folder_name for k in image_subdirs):
                    img_dir = child
                if mask_dir is None and any(k in folder_name for k in mask_subdirs):
                    mask_dir = child

    if img_dir is None or mask_dir is None:
        print(f"[Warning] Could not distinguish separate 'images' and 'GT' subfolders inside: {data_dir.resolve()}")
        return [], []

    print(f" Image Directory: {img_dir.resolve()}")
    print(f" Mask Directory:  {mask_dir.resolve()}")

    img_files = {f.stem: f for f in img_dir.glob("*") if f.suffix.lower() in valid_exts}
    mask_files = {f.stem: f for f in mask_dir.glob("*") if f.suffix.lower() in valid_exts}

    common_stems = sorted(list(set(img_files.keys()).intersection(set(mask_files.keys()))))
    paired_images = [img_files[stem] for stem in common_stems]
    paired_masks = [mask_files[stem] for stem in common_stems]

    print(f" Found {len(img_files)} images and {len(mask_files)} masks.")
    print(f" Successfully paired {len(paired_images)} image-mask samples.")
    
    return paired_images, paired_masks

def auto_discover_dataset_splits(root_dir):
    root_dir = Path(root_dir)
    splits = {}
    imgs, masks = discover_image_mask_pairs(root_dir)
    if imgs:
        splits["Default"] = (imgs, masks)
        return splits

    subdirs = [d for d in root_dir.rglob("*") if d.is_dir()]
    for d in subdirs:
        name_lower = d.name.lower()
        if any(keyword in name_lower for keyword in ["train", "training", "test", "testing", "val", "validation"]):
            has_subfolders = any(sub.is_dir() for sub in d.iterdir())
            if has_subfolders:
                imgs, masks = discover_image_mask_pairs(d)
                if imgs:
                    split_name = d.relative_to(root_dir).as_posix()
                    splits[split_name] = (imgs, masks)

    return splits

In [ ]:
# Execute Discovery on Project Data Directory
target_dir = CUSTOM_DATASET_PATH if CUSTOM_DATASET_PATH is not None else RAW_DATA_ROOT
print(f"Scanning directory for dataset splits: {target_dir.resolve()}\n" + "="*60)

dataset_splits = auto_discover_dataset_splits(target_dir)

if not dataset_splits:
    print("\n[Warning] No matching image-mask pairs found in RAW_DATA_ROOT.")
else:
    print("\nDataset Discovery Summary:")
    for split_name, (imgs, masks) in dataset_splits.items():
        print(f"  • Split '{split_name}': {len(imgs)} paired samples")

## 3. Mask Binarization & Boundary / Edge Map Extraction

In [54]:
def clean_and_binarize_mask(mask_input, threshold=128):
    if isinstance(mask_input, (str, Path)):
        mask = cv2.imread(str(mask_input), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise ValueError(f"Could not read mask file: {mask_input}")
    else:
        mask = mask_input.copy()
        if len(mask.shape) == 3:
            mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
    return (mask >= threshold).astype(np.uint8)

def compute_sobel_edge_map(binary_mask, kernel_size=3):
    mask_float = binary_mask.astype(np.float32)
    grad_x = cv2.Sobel(mask_float, cv2.CV_32F, 1, 0, ksize=kernel_size)
    grad_y = cv2.Sobel(mask_float, cv2.CV_32F, 0, 1, ksize=kernel_size)
    magnitude = cv2.magnitude(grad_x, grad_y)
    return (magnitude > 0.1).astype(np.uint8)

def apply_tactical_clahe(image_rgb, clip_limit=3.0, tile_grid_size=(8, 8)):
    lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

## 4. RGB Image-Only Augmentation Engine

In [55]:
def parse_tf_string(val):
    if hasattr(val, 'numpy'):
        val = val.numpy()
    if isinstance(val, bytes):
        return val.decode('utf-8')
    return str(val)

def get_rgb_only_augmentation_pipeline(image_size=(384, 384), is_train=True):
    height, width = image_size
    if is_train:
        return A.Compose([
            A.Resize(height=height, width=width),
            A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.4),
            A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
            A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=25, p=0.4),
            A.GaussianBlur(blur_limit=(3, 5), p=0.3),
            A.GaussNoise(p=0.3),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
        ])
    else:
        return A.Compose([
            A.Resize(height=height, width=width),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
        ])

def preprocess_sample_np(img_path_val, mask_path_val, image_size_h, image_size_w, is_train_val, compute_edge_val):
    img_path = parse_tf_string(img_path_val)
    mask_path = parse_tf_string(mask_path_val)
    is_train_bool = bool(parse_tf_string(is_train_val) == 'True' or is_train_val is True or is_train_val == 1)
    compute_edge_bool = bool(parse_tf_string(compute_edge_val) == 'True' or compute_edge_val is True or compute_edge_val == 1)
    target_size = (int(image_size_w), int(image_size_h))
    
    image = cv2.imread(img_path)
    if image is None:
        raise ValueError(f"Could not load image: {img_path}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    if COD_PREPROCESSING_MODE == 'RGB_CLAHE':
        image = apply_tactical_clahe(image)
        
    img_transform = get_rgb_only_augmentation_pipeline((int(image_size_h), int(image_size_w)), is_train=is_train_bool)
    image_np = img_transform(image=image)['image'].astype(np.float32)
    
    raw_mask = clean_and_binarize_mask(mask_path)
    mask_resized = cv2.resize(raw_mask, target_size, interpolation=cv2.INTER_NEAREST)
    mask_np = np.expand_dims(mask_resized.astype(np.float32), -1)
    
    if compute_edge_bool:
        edge_resized = compute_sobel_edge_map(mask_resized)
    else:
        edge_resized = np.zeros_like(mask_resized)
    edge_np = np.expand_dims(edge_resized.astype(np.float32), -1)
    
    filename = Path(img_path).name
    return image_np, mask_np, edge_np, filename

## 5. TensorFlow `tf.data.Dataset` Pipeline Factory

In [56]:
def create_tf_cod_dataset(image_paths, mask_paths, image_size=(384, 384), batch_size=16, is_train=True, compute_edge=True):
    img_str_paths = [str(Path(p).resolve()) for p in image_paths]
    mask_str_paths = [str(Path(p).resolve()) for p in mask_paths]
    
    dataset = tf.data.Dataset.from_tensor_slices((img_str_paths, mask_str_paths))
    if is_train:
        dataset = dataset.shuffle(buffer_size=len(img_str_paths), seed=SEED)
        
    def _map_fn(img_p, mask_p):
        img_tensor, mask_tensor, edge_tensor, fname = tf.py_function(
            func=preprocess_sample_np,
            inp=[img_p, mask_p, image_size[0], image_size[1], is_train, compute_edge],
            Tout=[tf.float32, tf.float32, tf.float32, tf.string]
        )
        img_tensor.set_shape([image_size[0], image_size[1], 3])
        mask_tensor.set_shape([image_size[0], image_size[1], 1])
        edge_tensor.set_shape([image_size[0], image_size[1], 1])
        fname.set_shape([])
        return img_tensor, mask_tensor, edge_tensor, fname
        
    dataset = dataset.map(_map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size, drop_remainder=False)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

## 6. Visual Inspection & Interactive Visualization

In [57]:
def denormalize_tf_image(img_tensor, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    img_np = img_tensor.numpy() if isinstance(img_tensor, tf.Tensor) else img_tensor
    img = (img_np * std + mean) * 255.0
    return np.clip(img, 0, 255).astype(np.uint8)

def create_mask_overlay(image_rgb, mask_binary, color=(255, 0, 0), alpha=0.5):
    overlay = image_rgb.copy()
    mask_bool = mask_binary.astype(bool)
    colored_mask = np.zeros_like(image_rgb)
    colored_mask[mask_bool] = color
    cv2.addWeighted(colored_mask, alpha, overlay, 1 - alpha, 0, overlay)
    return overlay

split_keys = list(dataset_splits.keys())
if split_keys:
    first_split = split_keys[0]
    img_paths, mask_paths = dataset_splits[first_split]
    
    print(f"Visualizing TensorFlow dataset batch (Mode: {COD_PREPROCESSING_MODE}) from split: '{first_split}'")
    tf_dataset = create_tf_cod_dataset(img_paths[:4], mask_paths[:4], image_size=IMAGE_SIZE, batch_size=4, is_train=True)
    
    for batch_imgs, batch_masks, batch_edges, batch_names in tf_dataset.take(1):
        fig, axes = plt.subplots(len(batch_imgs), 4, figsize=(16, 4 * len(batch_imgs)))
        if len(batch_imgs) == 1:
            axes = [axes]
            
        for i in range(len(batch_imgs)):
            img_rgb = denormalize_tf_image(batch_imgs[i])
            mask_np = batch_masks[i].numpy().squeeze(-1)
            edge_np = batch_edges[i].numpy().squeeze(-1)
            overlay_rgb = create_mask_overlay(img_rgb, mask_np)
            filename = batch_names[i].numpy().decode('utf-8')
            
            axes[i][0].imshow(img_rgb)
            axes[i][0].set_title(f"Augmented RGB Image\n({filename})")
            axes[i][0].axis("off")
            
            axes[i][1].imshow(mask_np, cmap="gray")
            axes[i][1].set_title("Clean GT Mask")
            axes[i][1].axis("off")
            
            axes[i][2].imshow(edge_np, cmap="inferno")
            axes[i][2].set_title("Clean Sobel Edge Map")
            axes[i][2].axis("off")
            
            axes[i][3].imshow(overlay_rgb)
            axes[i][3].set_title("Mask Overlay")
            axes[i][3].axis("off")
            
        plt.tight_layout()
        plt.show()
else:
    print("No dataset samples available to visualize.")

NameError: name 'COD_PREPROCESSING_MODE' is not defined

## 7. Automated Dataset Export Pipeline to `data/augmented/`

This pipeline generates $N$ augmented RGB variants per raw image (along with corresponding clean GT masks and Sobel edge maps) and exports them directly into the **`data/augmented/`** directory (`images/`, `GT/`, `edges/`).

In [ ]:
def generate_and_save_augmented_dataset(
    dataset_splits,
    output_root=AUGMENTED_DATA_ROOT,
    num_augmented_variants=3,
    image_size=IMAGE_SIZE
):
    """
    Generates N augmented RGB variants per image (along with corresponding clean GT masks and Sobel edge maps)
    and saves them directly into the data/augmented/ directory.
    """
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)
    print(f"Starting Dataset Augmentation & Export Pipeline...")
    print(f"  Target Output Directory:       {output_root.resolve()}")
    print(f"  Augmented Variants Per Sample: {num_augmented_variants}")
    
    for split_name, (img_paths, mask_paths) in dataset_splits.items():
        split_dir = output_root / split_name.replace("/", "_")
        img_out = split_dir / "images"
        gt_out = split_dir / "GT"
        edge_out = split_dir / "edges"
        
        img_out.mkdir(parents=True, exist_ok=True)
        gt_out.mkdir(parents=True, exist_ok=True)
        edge_out.mkdir(parents=True, exist_ok=True)
        
        print(f"\nProcessing split '{split_name}' ({len(img_paths)} original samples)...")
        
        for i in tqdm(range(len(img_paths))):
            stem = img_paths[i].stem
            
            # Load original RGB image & clean mask
            raw_img = cv2.imread(str(img_paths[i]))
            if raw_img is None:
                continue
            raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
            
            if COD_PREPROCESSING_MODE == 'RGB_CLAHE':
                raw_img = apply_tactical_clahe(raw_img)
                
            raw_mask = clean_and_binarize_mask(mask_paths[i])
            mask_resized = cv2.resize(raw_mask, image_size, interpolation=cv2.INTER_NEAREST)
            edge_map = compute_sobel_edge_map(mask_resized)
            
            # 1. Save Original Resized Sample
            orig_img_bgr = cv2.cvtColor(cv2.resize(raw_img, image_size), cv2.COLOR_RGB2BGR)
            cv2.imwrite(str(img_out / f"{stem}_orig.jpg"), orig_img_bgr)
            cv2.imwrite(str(gt_out / f"{stem}_orig.png"), mask_resized * 255)
            cv2.imwrite(str(edge_out / f"{stem}_orig.png"), edge_map * 255)
            
            # 2. Generate N Augmented RGB Variants
            img_aug_pipeline = get_rgb_only_augmentation_pipeline(image_size=image_size, is_train=True)
            
            for aug_idx in range(num_augmented_variants):
                aug_res = img_aug_pipeline(image=raw_img)
                aug_img_norm = aug_res['image']
                
                aug_img_uint8 = denormalize_tf_image(aug_img_norm)
                aug_img_bgr = cv2.cvtColor(aug_img_uint8, cv2.COLOR_RGB2BGR)
                
                aug_filename_img = f"{stem}_aug{aug_idx+1}.jpg"
                aug_filename_gt = f"{stem}_aug{aug_idx+1}.png"
                
                cv2.imwrite(str(img_out / aug_filename_img), aug_img_bgr)
                cv2.imwrite(str(gt_out / aug_filename_gt), mask_resized * 255)
                cv2.imwrite(str(edge_out / aug_filename_gt), edge_map * 255)
                
        print(f" Completed split '{split_name}'. Augmented files saved to: {split_dir.resolve()}")

# Execute the export pipeline on dataset splits:
if dataset_splits:
    generate_and_save_augmented_dataset(dataset_splits, output_root=AUGMENTED_DATA_ROOT, num_augmented_variants=2)